## 11_task_b — Job task: read practice_customers_clean, write practice_customers_enriched

Used by: practice-job-2-multitask (task b, depends on task a)
Config: CATALOG/SCHEMA/PARTITION/CONFIG_PATH widgets -> setup_config() (C360-style)
PARTITION: all | latest | prev | YYYY-MM-DD


In [ ]:

%run ../lib/config_lib


In [ ]:
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (OLD schema name — unchanged by migration)")
dbutils.widgets.text("PARTITION", "all", "all | latest | prev | YYYY-MM-DD")
dbutils.widgets.text("CONFIG_PATH", "", "Optional env config file (empty = built-in dev/stg/prd)")

CFG = setup_config(
    config_path=dbutils.widgets.get("CONFIG_PATH").strip(),
    run_date=dbutils.widgets.get("PARTITION").strip(),
    schema=dbutils.widgets.get("SCHEMA").strip(),
)


In [ ]:

from pyspark.sql import functions as F

src = CFG.table("practice_customers_clean")

df = (
    CFG.partition_filter(spark.read.table(src), table=src)
    .withColumn("annual_revenue", F.round(F.col("monthly_revenue") * 12, 2))
    .withColumn("high_value", F.col("monthly_revenue") > 500)
    .withColumn("risk_category",
        F.when(F.col("churn_risk") >= 0.66, F.lit("high"))
         .when(F.col("churn_risk") >= 0.33, F.lit("medium"))
         .otherwise(F.lit("low")))
)

CFG.write_partitioned(df, "practice_customers_enriched")

dst = CFG.table("practice_customers_enriched")
n = spark.read.table(dst).count()
print(f"TASK B DONE — wrote {n} rows to {dst} (PARTITION={CFG.run_date})")
